# Kezekshi: дообучение языковой модели (QLoRA)

Дообучаем **Qwen2.5-1.5B-Instruct** писать «Вероятную причину» инцидента по фактам, как в `src/llm/handler.py`.
Сравниваем базовую модель и дообученную на отложенной выборке и сохраняем модель в GGUF для Lambda `kezekshi-llm`.

**Перед запуском:** Среда выполнения → Сменить среду выполнения → **T4 GPU**. Потом Среда выполнения → Выполнить все.
Обучение идёт примерно 10–20 минут, сохранение в GGUF ещё около 10.

In [ ]:
%%capture
!pip install unsloth sacrebleu
!git clone -q https://github.com/littlelinli/ai-oncall-engineer.git repo

## 1. Датасет
Синтетические примеры из `ml/llm/make_dataset.py` (факты из сценариев Kezekshi, ответы из формулировок команды). Чтобы добавить настоящие инциденты, положите их в JSONL и передайте `--extra`.

In [ ]:
!python3 repo/ml/llm/make_dataset.py --n 900 --out data
import json
def load(p): return [json.loads(l) for l in open(p)]
train, val, test = load('data/train.jsonl'), load('data/val.jsonl'), load('data/test.jsonl')
print(len(train), len(val), len(test))
print(test[0]['messages'][1]['content'][:400])
print('->', test[0]['messages'][2]['content'])

## 2. Базовая модель (4 бита)

In [ ]:
from unsloth import FastLanguageModel
import torch
MAX_LEN = 1024
model, tokenizer = FastLanguageModel.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct', max_seq_length=MAX_LEN, load_in_4bit=True)

## 3. Метрики
Те же проверки, что в Lambda (`valid`), плюс: называет ли ответ правильную причину (ключевые слова) и близость к эталону (chrF).

In [ ]:
import re, sacrebleu
NUM = re.compile(r"\d+(?:[.,]\d+)?")
def numbers(t):
    out = set()
    for n in NUM.findall(t or ''):
        v = float(n.replace(',', '.')); out.add(round(v, 1)); out.add(float(round(v)))
    return out
def valid(text, lang, facts_text):
    if not text or len(text) < 20: return 'empty'
    if len(text) > 600: return 'too long'
    letters = [c for c in text if c.isalpha()]
    cyr = sum(1 for c in letters if 'а' <= c.lower() <= 'я' or c.lower() == 'ё') / max(len(letters), 1)
    if lang == 'ru' and cyr < 0.6: return 'not russian'
    if lang == 'en' and cyr > 0.05: return 'not english'
    if [n for n in numbers(text) if n not in numbers(facts_text)]: return 'invented numbers'
    if any(m in text for m in ('```', '**', '##')): return 'markdown'
    return None
CAUSE_WORDS = {
 'traffic_surge': (['посетител','запрос','наплыв','трафик'], ['traffic','request','visitor','surge']),
 'background_process': (['фонов'], ['background']),
 'memory_leak': (['утечк'], ['leak']),
 'memory_pressure': (['памят'], ['memory']),
 'log_growth': (['лог'], ['log']),
 'files_accumulate': (['файл'], ['file']),
 'disk_jump': (['файл','скач','резк'], ['file','sudden','jump']),
 'db_lock': (['баз','блокир'], ['database','lock']),
 'upstream_slow': (['платеж','внешн','оплат'], ['payment','external']),
 'dependency_down': (['баз','подключ','недоступ'], ['database','connect','unavailable']),
 'overload': (['перегруж','очеред','таймаут'], ['overload','queue','time']),
 'app_down': (['приложен'], ['application']),
 'bad_deploy': (['обновлен','деплой','код'], ['deploy','update','code']),
 'cpu_saturation': (['процессор','cpu'], ['cpu']),
 'slow_dependency': (['баз','внешн','зависим'], ['database','external','depend']),
 'brute_force': (['парол'], ['password']),
 'verbose_logging': (['debug','подробн'], ['debug','verbose']),
 'error_loop': (['ошибк'], ['error']),
 'server_down': (['агент','сервер'], ['agent','server']),
}
def cause_ok(text, cause, lang):
    t = text.lower().replace('ё', 'е')
    words = CAUSE_WORDS[cause][0 if lang == 'ru' else 1]
    return any(w in t for w in words)

def generate(ex, max_new_tokens=160):
    msgs = ex['messages'][:2]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, temperature=0.3, top_p=0.9, do_sample=True,
                         pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip().strip('"')
    return re.sub(r'^(Вероятная причина|Причина|Likely cause|Cause)\s*:\s*', '', text, flags=re.I)

def evaluate(name, examples):
    FastLanguageModel.for_inference(model)
    rows = []
    for ex in examples:
        lang, cause = ex['meta']['lang'], ex['meta']['cause']
        out = generate(ex)
        why = valid(out, lang, ex['messages'][1]['content'])
        rows.append({'lang': lang, 'cause': cause, 'output': out, 'reference': ex['messages'][2]['content'],
                     'passes_checks': why is None, 'reject_reason': why or '', 'right_cause': cause_ok(out, cause, lang)})
    n = len(rows)
    chrf = sacrebleu.corpus_chrf([r['output'] for r in rows], [[r['reference'] for r in rows]]).score
    summary = {'model': name, 'passes_checks_%': round(100 * sum(r['passes_checks'] for r in rows) / n, 1),
               'right_cause_%': round(100 * sum(r['right_cause'] for r in rows) / n, 1),
               'chrF': round(chrf, 1), 'avg_chars': round(sum(len(r['output']) for r in rows) / n)}
    print(summary)
    return summary, rows

## 4. Как пишет базовая модель (до обучения)

In [ ]:
base_summary, base_rows = evaluate('base', test)
for r in base_rows[:3]: print(r['lang'], r['cause'], '|', r['output'])

## 5. QLoRA: обучаем только адаптеры (около 1% параметров)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=7)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
def to_text(rows): return Dataset.from_list([{'text': tokenizer.apply_chat_template(r['messages'], tokenize=False)} for r in rows])
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=to_text(train), eval_dataset=to_text(val),
    args=SFTConfig(dataset_text_field='text', max_seq_length=MAX_LEN, per_device_train_batch_size=8,
                   gradient_accumulation_steps=2, num_train_epochs=2, learning_rate=2e-4, warmup_steps=10,
                   lr_scheduler_type='cosine', logging_steps=10, eval_strategy='epoch', save_strategy='no',
                   fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
                   optim='adamw_8bit', seed=7, output_dir='out', report_to='none'))
# считаем потери только на ответе модели, не на фактах
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
stats = trainer.train()
print(stats)

## 6. Дообученная модель: те же метрики

In [ ]:
ft_summary, ft_rows = evaluate('fine-tuned', test)
import pandas as pd
display(pd.DataFrame([base_summary, ft_summary]).set_index('model'))
pd.DataFrame([{'cause': b['cause'], 'lang': b['lang'], 'base': b['output'], 'fine_tuned': f['output'], 'reference': b['reference']}
              for b, f in zip(base_rows, ft_rows)]).to_csv('comparison.csv', index=False)
for b, f in list(zip(base_rows, ft_rows))[:5]:
    print(b['cause'], b['lang']); print('  base:', b['output']); print('  ft:  ', f['output']); print()

## 7. Сохранение
1. LoRA-адаптер (маленький, для отчёта и повторного обучения).
2. Модель целиком в GGUF Q4_K_M (около 1 ГБ) для Lambda `kezekshi-llm`.

In [ ]:
VERSION = 'kezekshi-qwen2.5-1.5b-ft-v1'
model.save_pretrained(VERSION + '-lora'); tokenizer.save_pretrained(VERSION + '-lora')
model.save_pretrained_gguf(VERSION, tokenizer, quantization_method='q4_k_m')
import glob, os, shutil
gguf = [p for p in glob.glob(f'{VERSION}*/*.gguf') + glob.glob('*.gguf') if 'q4_k_m' in p.lower()][0]
final = f'{VERSION}-q4_k_m.gguf'
shutil.move(gguf, final)
print(final, round(os.path.getsize(final) / 1e9, 2), 'GB')

## 8. Отдать модель в Kezekshi
1. Скачайте файл слева (значок папки → `kezekshi-qwen2.5-1.5b-ft-v1-q4_k_m.gguf` → три точки → Скачать) и `comparison.csv`.
2. AWS Console → S3 → `kezekshi-dataset-788383723111` → папка `models/llm/` → Upload этого файла.
3. В CloudShell в `terraform/envs/dev.tfvars` добавить строку
   `llm_model_key = "models/llm/kezekshi-qwen2.5-1.5b-ft-v1-q4_k_m.gguf"` и сделать `terraform apply`.

Таблица из шага 6 и `comparison.csv` это результат для диплома: базовая модель против дообученной.

In [ ]:
from google.colab import files
files.download('comparison.csv')